# Microsoft 3B — What Makes a GitHub Repository Worth Trusting?
## Milestone 2: testing the evidence

Our goal is to understand what evidence helps us place confidence in a repository. A popular or old project can look convincing, but those clues can also mislead us. This notebook checks which signals separate our selected groups, and whether that separation survives harder comparisons.

We use a saved copy of **166 repositories**: **112 known-good** and **54 comparison examples**. A saved copy, or **snapshot**, lets us rerun the analysis on the same data instead of having the numbers change every time GitHub changes.

| Group | Repositories | Why they are included |
|---|---:|---|
| Original known-good | 82 | Starting reference projects |
| Original uncertain | 24 | Projects whose trustworthiness is not established here |
| Teammate-added known-good | 30 | Additional reference projects |
| Maintenance counterexamples | 30 | Older projects selected for deprecation, archival, or maintenance concerns |

**Important:** `label = 1` means known-good; `label = 0` means a comparison example. Zero does **not** mean malicious. An archived project can still be legitimate and useful.

### How to use this notebook
Run the cells from top to bottom, or use **Runtime → Run all** in Colab. The notebook loads your saved Drive files. You can also upload `repositories.csv` and `snapshot_notes.json` to the Colab session. No GitHub token is needed.

The original Milestone 1 notebook preserves the earlier experiments. This notebook is the working version for the next phase.

**Data version:** `20261006T203318233434Z`.

### A few terms we will use
- **Repository (repo):** a project's files and development history on GitHub.
- **Feature or signal:** a measurable clue, such as age, stars, or time since the last push.
- **EDA:** exploratory data analysis—looking for patterns and checking whether they have reasonable explanations.
- **Confounding:** another factor, such as age, can make a signal look meaningful when it is mostly reflecting how our dataset was assembled.
- **Temporal:** measured over time rather than at one moment.

The next cell imports the analysis tools and lists the signals we will examine.


In [ ]:
# Tools for tables, charts, and AUC comparisons.
import json
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_auc_score

sns.set_theme(style="whitegrid")
SNAPSHOT_ID = "20261006T203318233434Z"
SIGNALS = [
    "stargazers_count",
    "forks_count",
    "subscribers_count",
    "network_count",
    "open_issues_count",
    "size",
    "description_length",
    "topics_count",
    "repo_age_days",
    "days_since_push",
    "star_fork_ratio",
    "stars_per_day",
]

## 1. Load the saved data
We load the repository table and its notes, then restore date columns to a format Python can compare.

`repo_features` is our main table. Each row is one repository; each column is a measurement or a note about that repository.

All time-based values use the snapshot's saved **analysis reference time**. We do not recalculate them against today's date. This keeps the results reproducible.


In [ ]:
# Find the saved files in Drive or the current session.
drive_snapshot = Path("/content/drive/MyDrive/Microsoft_3B/snapshots") / SNAPSHOT_ID
candidates = [drive_snapshot, Path("/content"), Path.cwd(), Path.cwd() / "upload"]
data_dir = next(
    (p for p in candidates if (p / "repositories.csv").exists() and (p / "snapshot_notes.json").exists()), None
)
if data_dir is None:
    try:
        import importlib

        drive = importlib.import_module("google.colab.drive")
    except ImportError:
        raise FileNotFoundError("Place repositories.csv and snapshot_notes.json in the working directory.") from None
    drive.mount("/content/drive")
    data_dir = drive_snapshot
if not all((data_dir / name).exists() for name in ["repositories.csv", "snapshot_notes.json"]):
    raise FileNotFoundError("Snapshot files missing. Upload both files to /content and rerun this cell.")
repo_features = pd.read_csv(data_dir / "repositories.csv")
snapshot_notes = json.loads((data_dir / "snapshot_notes.json").read_text())
for column in ["created_at", "pushed_at", "analysis_as_of", "license_fetched_at"]:
    repo_features[column] = pd.to_datetime(repo_features[column], utc=True, errors="raise")
analysis_as_of = pd.Timestamp(snapshot_notes["analysis_as_of"])
print("Snapshot:", snapshot_notes["snapshot_id"])
print("Analysis reference:", analysis_as_of)

## 2. Check the data before trusting the results
This cell checks that:
- We have 166 unique repository names and the expected counts in each source group.
- Labels match their source groups, and maintenance counterexamples have a recorded selection reason.
- Every row uses the same analysis reference time.
- The saved age, push recency, and star ratios match their formulas.
- License values and collection timestamps are present.

An **assertion** is a check that stops the notebook when an assumption fails. Stopping here is better than producing a convincing chart from inconsistent data.

Repository names are compared in lowercase so capitalization does not create false duplicates. Future collection should also preserve GitHub's repository ID and both the requested and returned names, because projects can be renamed or transferred.

**Source and selection reason explain our dataset. They should not become model inputs:** otherwise the model could learn our labeling process instead of useful repository evidence.

The output shows group sizes, detected licenses, and missing values. Missing language or selection reasons outside the counterexample group can be expected; missing data should be interpreted by column.


In [ ]:
# Stop if the snapshot no longer matches our documented assumptions.
assert snapshot_notes["snapshot_id"] == SNAPSHOT_ID
assert len(repo_features) == snapshot_notes["repository_count"] == 166
repo_keys = repo_features["full_name"].astype("string").str.strip().str.lower()
assert repo_keys.notna().all()
assert repo_keys.is_unique
assert repo_features["dataset_source"].value_counts().to_dict() == snapshot_notes["cohort_counts"]
expected_labels = repo_features["dataset_source"].map(
    {"original_good": 1, "teammate_good": 1, "original_uncertain": 0, "maintenance_counterexample": 0}
)
assert expected_labels.notna().all()
assert repo_features["label"].eq(expected_labels).all()
assert (
    repo_features.loc[repo_features["dataset_source"].eq("maintenance_counterexample"), "negative_reason"].notna().all()
)
assert repo_features["analysis_as_of"].eq(analysis_as_of).all()
assert repo_features["license_fetched_at"].notna().all()
expected_features = {
    "repo_age_days": (analysis_as_of - repo_features["created_at"]).dt.total_seconds() / 86400,
    "days_since_push": (analysis_as_of - repo_features["pushed_at"]).dt.total_seconds() / 86400,
    "star_fork_ratio": repo_features["stargazers_count"] / (repo_features["forks_count"] + 1),
}
expected_features["stars_per_day"] = repo_features["stargazers_count"] / expected_features["repo_age_days"].clip(
    lower=1
)
for name, values in expected_features.items():
    assert np.allclose(repo_features[name], values, rtol=1e-9, atol=1e-9), name
assert repo_features["repo_age_days"].ge(0).all()
assert repo_features["days_since_push"].ge(0).all()
assert repo_features["has_license"].isin([True, False]).all()
display(
    repo_features.groupby("dataset_source").agg(
        repositories=("full_name", "size"), detected_licenses=("has_license", "sum")
    )
)
display(repo_features.isna().sum().loc[lambda s: s > 0].rename("missing_values").to_frame())
print("Snapshot validation passed.")

## 3. Which individual clues separate our groups?
We compare each signal on its own. This is exploratory analysis, not a trained model or a test of performance on new repositories.

### How to read the table
| Column | Meaning |
|---|---|
| `n_good`, `n_uncertain` | Number of usable measurements in each group |
| `good_median`, `uncertain_median` | Typical value in each group; the median is the middle value |
| `raw_auc` | How often a randomly selected known-good repo has a higher signal value than a randomly selected comparison repo, with ties counting halfway |
| `separation_auc` | The same comparison with direction folded away using `max(AUC, 1 − AUC)` |

**AUC** stands for *area under the ROC curve*. For raw AUC, **0.5** means no ranking advantage, values above 0.5 point toward higher values in the good group, and values below 0.5 point toward lower values in that group.

For example, a low raw AUC for `days_since_push` can be sensible: recently active projects have **fewer** days since their last push. Separation AUC makes this easier to compare with signals where higher values are associated with good.

**A high AUC here is not a trust score or a probability that a repository is good.** It can reflect our selection choices. We keep raw AUC so we can see whether a signal's direction changes.

The function below gives every signal the same treatment and reports how many rows were used.


In [ ]:
# Apply the same comparison to every signal; skip unavailable measurements.
def evaluate_signals(data, signals=SIGNALS):
    rows = []
    for signal in signals:
        valid = data[[signal, "label"]].replace([np.inf, -np.inf], np.nan).dropna()
        good = valid.loc[valid["label"].eq(1), signal]
        uncertain = valid.loc[valid["label"].eq(0), signal]
        auc = roc_auc_score(valid["label"], valid[signal]) if len(good) and len(uncertain) else np.nan
        rows.append(
            {
                "signal": signal,
                "n_good": len(good),
                "n_uncertain": len(uncertain),
                "good_median": good.median(),
                "uncertain_median": uncertain.median(),
                "raw_auc": auc,
                "separation_auc": max(auc, 1 - auc) if pd.notna(auc) else np.nan,
            }
        )
    return pd.DataFrame(rows).sort_values("separation_auc", ascending=False).reset_index(drop=True)


baseline_results = evaluate_signals(repo_features)
display(baseline_results.round(3))
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=baseline_results, x="separation_auc", y="signal", ax=ax)
ax.axvline(0.5, color="black", linestyle="--")
ax.set(
    xlim=(0.45, 1),
    title="How strongly does each clue separate our selected groups?",
    xlabel="Separation AUC (0.5 = no ranking advantage)",
    ylabel="Repository signal",
)
plt.tight_layout()
plt.show()

## 4. Does “old = good” survive harder examples?
Our original dataset contained mostly older known-good repos and younger uncertain repos. Age could therefore look excellent while mainly reflecting which projects we selected.

We compare three versions of the dataset at the same reference time:
1. **106 repos:** the original 82 good and 24 uncertain.
2. **136 repos:** add 30 teammate-selected good projects.
3. **166 repos:** also add 30 maintenance counterexamples.

The key question is whether age still separates the groups after older comparison projects are included. We also check push recency.

These versions overlap; they are **not three independent test sets**. This is a stress test of our interpretation, not proof of performance on arbitrary GitHub projects.


In [ ]:
# Keep the three dataset versions separate so selection effects are visible.
cohort_views = {
    "original_106": repo_features[repo_features["dataset_source"].isin(["original_good", "original_uncertain"])],
    "expanded_good_136": repo_features[repo_features["dataset_source"].ne("maintenance_counterexample")],
    "full_166": repo_features,
}
composition_results = pd.concat(
    [evaluate_signals(view).assign(dataset=name) for name, view in cohort_views.items()], ignore_index=True
)
display(
    composition_results.loc[
        composition_results["signal"].isin(["repo_age_days", "days_since_push"]),
        ["dataset", "signal", "n_good", "n_uncertain", "raw_auc", "separation_auc"],
    ].round(3)
)
display(
    repo_features.groupby("dataset_source")[["repo_age_days", "days_since_push", "stargazers_count"]].median().round(3)
)

## 5. Compare repositories of roughly similar ages
Instead of comparing every old project with every young project, we group repositories into four age ranges and repeat the analysis within each range. This is called **age-stratified analysis**.

It reduces some age differences, but does not remove them completely. A two-year project and a nearly five-year project still share a bin. Selection based on maintenance also remains a concern.

Start with the counts: a result based on only a few known-good projects is less dependable. Then read the push-recency table and heatmap.

The heatmap uses **raw AUC**: 0.5 is the midpoint. A value near zero can indicate strong separation in the opposite direction—not a useless feature. Always check which direction makes sense.


In [ ]:
# Compare within age ranges instead of relying only on the full dataset.
age_view = repo_features.copy()
age_view["age_group"] = pd.cut(
    age_view["repo_age_days"],
    [0, 730, 1825, 3650, np.inf],
    labels=["<2 years", "2-5 years", "5-10 years", "10+ years"],
    include_lowest=True,
)
display(pd.crosstab(age_view["age_group"], age_view["label"]).rename(columns={0: "uncertain", 1: "good"}))
age_results = pd.concat(
    [
        evaluate_signals(group, [s for s in SIGNALS if s != "repo_age_days"]).assign(age_group=str(name))
        for name, group in age_view.groupby("age_group", observed=True)
    ],
    ignore_index=True,
)
display(age_results.loc[age_results["signal"].eq("days_since_push")].round(3))
age_pivot = age_results.pivot(index="signal", columns="age_group", values="raw_auc")
age_pivot = age_pivot.reindex(columns=["<2 years", "2-5 years", "5-10 years", "10+ years"])
plt.figure(figsize=(9, 6))
sns.heatmap(age_pivot, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="vlag", center=0.5)
plt.title("Do the patterns hold among similarly aged repositories?")
plt.xlabel("Repository age group")
plt.ylabel("Repository signal")
plt.tight_layout()
plt.show()

## 6. Are we measuring owner identity or counting popularity twice?
### Check A: remove Microsoft-owned repositories
We repeat the signal comparisons without Microsoft-owned projects. If the results stay similar, Microsoft alone does not explain the pattern.

That does not rule out broader organization bias. Projects from the same owner may share practices or history. When the modeling owner evaluates generalization, keeping an owner's projects together in a train/test split can help test whether performance transfers to other owners.

Here, owner names help us investigate the dataset; they are not proposed trust predictors.

### Check B: compare popularity signals
Stars, forks, subscribers, and network count may overlap. **Correlation** measures how two values move together. Pearson correlation ranges from −1 to +1; +1 means an exact positive linear relationship in the observed data.

Strong correlation suggests repeated information, not several independent votes for trust. Large projects can also strongly influence Pearson correlation. We keep forks and network count for comparison, but should avoid treating both as separate evidence when they repeat the same measurement.


In [ ]:
# First check owner sensitivity, then check overlapping popularity measures.
owners = repo_features["full_name"].str.split("/").str[0].str.lower()
owner_table = pd.crosstab(owners, repo_features["label"]).rename(columns={0: "uncertain", 1: "good"})
owner_table["total"] = owner_table.sum(axis=1)
display(owner_table.sort_values("total", ascending=False).head(15))
without_microsoft_results = evaluate_signals(repo_features.loc[owners.ne("microsoft")])
owner_results = baseline_results[["signal", "raw_auc", "separation_auc"]].merge(
    without_microsoft_results[["signal", "raw_auc", "separation_auc"]],
    on="signal",
    suffixes=("_all", "_without_microsoft"),
)
display(owner_results.round(3))
popularity_columns = ["stargazers_count", "forks_count", "subscribers_count", "network_count"]
popularity_corr = repo_features[popularity_columns].corr()
display(popularity_corr.round(3))
print("Rows with equal fork/network counts:", repo_features["forks_count"].eq(repo_features["network_count"]).sum())
plt.figure(figsize=(6, 5))
sns.heatmap(popularity_corr, annot=True, fmt=".3f", vmin=-1, vmax=1, cmap="vlag", center=0)
plt.title("Are popularity signals repeating the same information?")
plt.tight_layout()
plt.show()

## 7. What can we responsibly say?
**Our clearest finding is that dataset choices matter.** Age looked exceptionally strong until we added older maintenance counterexamples. A useful signal should survive difficult examples, not just easy comparisons.

Recent push activity is promising, but many counterexamples were selected because development had slowed or stopped. That makes the activity result partly tied to how we built the comparison group. We still need quiet legitimate projects and active uncertain projects to challenge it.

### What these measurements do—and do not—tell us
| Measurement | What it tells us | What it does not establish |
|---|---|---|
| Repository age | How long the repository has existed | Whether it deserves trust |
| Last push | Time since the recorded push | Meaningful work, steady development, or community participation |
| Stars per day | Lifetime stars divided by age, with a one-day denominator floor | When a spike happened or how growth changed over time |
| Detected license | Whether GitHub returned a license key | Security, license suitability, or overall trust |
| Correlated popularity counts | Some signals may repeat information | Independent confirmation of quality |

The original 25-repo contributor experiment is preserved in Milestone 1. Its capped counts and failed-request fallback make it unsuitable for current features.

We have not established a calibrated trust probability or a repository-health score. The results below describe **this saved dataset**; earlier Milestone 1 values describe earlier experiments.

The next cell displays the original snapshot notes so the collection limitations remain visible.


In [ ]:
# Show the collection notes exactly as recorded with the snapshot.
print("Recorded snapshot limitations:")
for limitation in snapshot_notes["limitations"]:
    print("•", limitation)
print("\nFeature definitions:")
display(pd.Series(snapshot_notes["feature_definitions"], name="definition").to_frame())

## 9. Viral repository pilot: what happened after the spike?

We now have two pilot examples: **OpenClaw** and **DeepSeek-R1**. The question is whether a large burst of stars was followed by continued development. Stars measure attention; commits measure one kind of activity. Neither establishes trustworthiness.

We look at four weeks before the largest observed complete star week, the peak week itself, and eight weeks afterward. This is a retrospective analysis: the eight-week result becomes available only after that follow-up period.

Run the sections below to review saved results. They make no GitHub requests and do not overwrite your collected files. If the saved Drive files are unavailable, the notebook uses the exact event-window tables preserved in the uploaded notebook's outputs and announces that fallback. Those excerpts are not the full historical dataset.


In [ ]:
# Locations of the already collected pilot files.
TEMPORAL_ROOT = Path("/content/drive/MyDrive/Microsoft_3B/temporal")
PILOTS = {
    "openclaw/openclaw": {
        "folder": "openclaw_openclaw/20261007T005052024078Z",
        "file": "peak_event_window.csv",
        "peak": "2026-01-25",
        "method": "GitHub weekly commit statistics",
        "created": None,  # Creation date was not included in the displayed pilot outputs.
    },
    "deepseek-ai/DeepSeek-R1": {
        "folder": "deepseek-ai_DeepSeek-R1/20261007T010911807574Z",
        "file": "event_window.csv",
        "peak": "2025-01-26",
        "method": "Default-branch non-merge commits, committer date",
        "created": "2025-01-20T11:57:28Z",
    },
}
# These values are copied from saved notebook outputs, not newly collected.
PRESERVED_EVENT_TABLES = {
    "openclaw/openclaw": [
        {
            "weeks_from_peak": -4,
            "week_display_utc": "2025-12-28 00:00:00+00:00",
            "weekly_stars": 632,
            "weekly_commits": 565,
        },
        {
            "weeks_from_peak": -3,
            "week_display_utc": "2026-01-04 00:00:00+00:00",
            "weekly_stars": 1647,
            "weekly_commits": 1891,
        },
        {
            "weeks_from_peak": -2,
            "week_display_utc": "2026-01-11 00:00:00+00:00",
            "weekly_stars": 1671,
            "weekly_commits": 1619,
        },
        {
            "weeks_from_peak": -1,
            "week_display_utc": "2026-01-18 00:00:00+00:00",
            "weekly_stars": 5515,
            "weekly_commits": 1521,
        },
        {
            "weeks_from_peak": 0,
            "week_display_utc": "2026-01-25 00:00:00+00:00",
            "weekly_stars": 116286,
            "weekly_commits": 815,
        },
        {
            "weeks_from_peak": 1,
            "week_display_utc": "2026-02-01 00:00:00+00:00",
            "weekly_stars": 37718,
            "weekly_commits": 674,
        },
        {
            "weeks_from_peak": 2,
            "week_display_utc": "2026-02-08 00:00:00+00:00",
            "weekly_stars": 19790,
            "weekly_commits": 1544,
        },
        {
            "weeks_from_peak": 3,
            "week_display_utc": "2026-02-15 00:00:00+00:00",
            "weekly_stars": 21252,
            "weekly_commits": 3081,
        },
        {
            "weeks_from_peak": 4,
            "week_display_utc": "2026-02-22 00:00:00+00:00",
            "weekly_stars": 23805,
            "weekly_commits": 2327,
        },
        {
            "weeks_from_peak": 5,
            "week_display_utc": "2026-03-01 00:00:00+00:00",
            "weekly_stars": 34983,
            "weekly_commits": 2367,
        },
        {
            "weeks_from_peak": 6,
            "week_display_utc": "2026-03-08 00:00:00+00:00",
            "weekly_stars": 35760,
            "weekly_commits": 1785,
        },
        {
            "weeks_from_peak": 7,
            "week_display_utc": "2026-03-15 00:00:00+00:00",
            "weekly_stars": 15779,
            "weekly_commits": 1893,
        },
        {
            "weeks_from_peak": 8,
            "week_display_utc": "2026-03-22 00:00:00+00:00",
            "weekly_stars": 11026,
            "weekly_commits": 2675,
        },
    ],
    "deepseek-ai/DeepSeek-R1": [
        {
            "weeks_from_peak": -1,
            "week_display_utc": "2025-01-19 00:00:00+00:00",
            "weekly_stars": 20090,
            "weekly_commits": 7,
        },
        {
            "weeks_from_peak": 0,
            "week_display_utc": "2025-01-26 00:00:00+00:00",
            "weekly_stars": 31948,
            "weekly_commits": 5,
        },
        {
            "weeks_from_peak": 1,
            "week_display_utc": "2025-02-02 00:00:00+00:00",
            "weekly_stars": 13147,
            "weekly_commits": 4,
        },
        {
            "weeks_from_peak": 2,
            "week_display_utc": "2025-02-09 00:00:00+00:00",
            "weekly_stars": 6613,
            "weekly_commits": 3,
        },
        {
            "weeks_from_peak": 3,
            "week_display_utc": "2025-02-16 00:00:00+00:00",
            "weekly_stars": 4236,
            "weekly_commits": 3,
        },
        {
            "weeks_from_peak": 4,
            "week_display_utc": "2025-02-23 00:00:00+00:00",
            "weekly_stars": 3865,
            "weekly_commits": 3,
        },
        {
            "weeks_from_peak": 5,
            "week_display_utc": "2025-03-02 00:00:00+00:00",
            "weekly_stars": 1417,
            "weekly_commits": 0,
        },
        {
            "weeks_from_peak": 6,
            "week_display_utc": "2025-03-09 00:00:00+00:00",
            "weekly_stars": 989,
            "weekly_commits": 0,
        },
        {
            "weeks_from_peak": 7,
            "week_display_utc": "2025-03-16 00:00:00+00:00",
            "weekly_stars": 698,
            "weekly_commits": 0,
        },
        {
            "weeks_from_peak": 8,
            "week_display_utc": "2025-03-23 00:00:00+00:00",
            "weekly_stars": 614,
            "weekly_commits": 0,
        },
    ],
}
event_tables = {}
for repository, settings in PILOTS.items():
    folder, filename = settings["folder"], settings["file"]
    saved_file = TEMPORAL_ROOT / folder / filename if isinstance(folder, str) and isinstance(filename, str) else None
    if saved_file is not None and saved_file.exists():
        table = pd.read_csv(saved_file)
        source = "saved Drive CSV"
    else:
        table = pd.DataFrame(PRESERVED_EVENT_TABLES[repository])
        source = "preserved notebook event-window output (fallback)"
    table["week_display_utc"] = pd.to_datetime(table["week_display_utc"], utc=True)
    peak_date = settings["peak"]
    if not isinstance(peak_date, str):
        raise ValueError("Pilot peak date is required")
    peak = pd.Timestamp(peak_date, tz="UTC")
    table["weeks_from_peak"] = (table["week_display_utc"] - peak).dt.days // 7
    table = table.loc[table["weeks_from_peak"].between(-4, 8)].sort_values("week_display_utc")
    assert not table["week_display_utc"].duplicated().any(), "Duplicate weeks: inspect the saved data."
    for column in ["weekly_stars", "weekly_commits"]:
        table[column] = pd.to_numeric(table[column], errors="raise")
        assert table[column].dropna().ge(0).all(), "Counts must be nonnegative."
    event_tables[repository] = table.reset_index(drop=True)
    print(f"{repository}: {len(table)} event weeks; source: {source}")

## 10. Summarize activity without inventing missing weeks

An **active week** has at least one counted commit. The **post/pre ratio** compares median weekly commits after and before the peak; values above 1 mean the median increased.

Missing coverage stays missing. DeepSeek-R1 was created on January 20, 2025, so it could not have a full four-week baseline before its January 26 peak. Its post-peak activity can still be described, but its baseline ratio is unavailable. OpenClaw's ratio is retained as a pilot result; its creation-date check remains unverified here.


In [ ]:
def summarize_event(repository, table, settings):
    pre = table.loc[table["weeks_from_peak"].between(-4, -1)]
    post = table.loc[table["weeks_from_peak"].between(1, 8)]
    pre_complete = set(pre["weeks_from_peak"]) == set(range(-4, 0)) and pre["weekly_commits"].notna().all()
    post_complete = set(post["weeks_from_peak"]) == set(range(1, 9)) and post["weekly_commits"].notna().all()
    peak_date = settings["peak"]
    if not isinstance(peak_date, str):
        raise ValueError("Pilot peak date is required")
    peak = pd.Timestamp(peak_date, tz="UTC")
    created = pd.Timestamp(settings["created"]) if settings["created"] else None
    existed_for_baseline = created is None or created <= peak - pd.Timedelta(weeks=4)
    pre_median = pre["weekly_commits"].median() if pre_complete and existed_for_baseline else np.nan
    post_median = post["weekly_commits"].median() if post_complete else np.nan
    ratio = post_median / pre_median if pd.notna(pre_median) and pre_median > 0 and post_complete else np.nan
    if not existed_for_baseline:
        status = "Unavailable: repository did not exist for the full baseline"
    elif not pre_complete or not post_complete:
        status = "Unavailable: incomplete coverage"
    elif pre_median == 0:
        status = "Unavailable: zero baseline median"
    else:
        status = "Pilot only: creation date unverified" if created is None else "Available"
    peak_rows = table.loc[table["weeks_from_peak"].eq(0), "weekly_stars"]
    assert len(peak_rows) == 1, "Expected exactly one peak week."
    return {
        "full_name": repository,
        "peak_week": peak,
        "peak_week_stars": int(peak_rows.iloc[0]),
        "pre_weeks_observed": len(pre),
        "post_weeks_observed": len(post),
        "post_active_week_share": post["weekly_commits"].gt(0).mean() if post_complete else np.nan,
        "pre_median_weekly_commits": pre_median,
        "post_median_weekly_commits": post_median,
        "post_pre_commit_ratio": ratio,
        "ratio_status": status,
        "commit_method": settings["method"],
    }


pilot_summary = pd.DataFrame([summarize_event(repo, event_tables[repo], settings) for repo, settings in PILOTS.items()])
display(pilot_summary.round(3))

## 11. Read the two timelines

Each repository gets separate star and commit charts, with its own scale. Week 0 is the peak. The counts describe activity, not code quality, and the commit methods differ between repositories.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for row, (repository, table) in enumerate(event_tables.items()):
    for column, (field, label) in enumerate([("weekly_stars", "Stars gained"), ("weekly_commits", "Commits counted")]):
        ax = axes[row, column]
        ax.plot(table["weeks_from_peak"], table[field], marker="o")
        ax.axvline(0, color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"{repository}: {label.lower()}")
        ax.set_xlabel("Weeks from star peak")
        ax.set_ylabel(label)
        ax.grid(alpha=0.2)
fig.suptitle("Viral pilot: attention and development around the peak")
fig.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import requests

REA_REPO = "morluto/rea"

# Reuse your existing authenticated session if one is available.
rea_session = globals().get("session")
if not isinstance(rea_session, requests.Session):
    rea_session = requests.Session()

response = rea_session.get(
    f"https://api.github.com/repos/{REA_REPO}",
    headers={"Accept": "application/vnd.github+json"},
    timeout=30,
)

print("HTTP status:", response.status_code)
response.raise_for_status()
rea_metadata = response.json()

assert rea_metadata["full_name"].lower() == REA_REPO.lower()
assert not rea_metadata["fork"], "This is a fork; inspect its parent first."

rea_collected_at = pd.Timestamp.now(tz="UTC")

display(
    pd.DataFrame(
        [
            {
                "full_name": rea_metadata["full_name"],
                "created_at": pd.to_datetime(rea_metadata["created_at"], utc=True),
                "collected_at": rea_collected_at,
                "stars_now": rea_metadata["stargazers_count"],
                "forks_now": rea_metadata["forks_count"],
                "default_branch": rea_metadata["default_branch"],
                "archived": rea_metadata["archived"],
            }
        ]
    )
)

In [ ]:
# Collect every available page of REA's weekly star history.
rea_star_pages = []
rea_star_records = []
url = f"https://api.github.com/repos/{REA_REPO}/stargazers/history"

while url:
    result = rea_session.get(
        url,
        headers={
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2026-03-10",
        },
        timeout=30,
    )
    result.raise_for_status()
    records = result.json()

    assert isinstance(records, list), "Unexpected history format."
    rea_star_pages.append({"url": result.url, "response": records})
    rea_star_records.extend(records)
    url = result.links.get("next", {}).get("url")

rea_stars = pd.DataFrame(rea_star_records)
assert not rea_stars.empty, "No star history returned."
assert {"week", "total", "days"}.issubset(rea_stars.columns)
assert not rea_stars["week"].duplicated().any(), "Duplicate weeks."
assert rea_stars["days"].apply(len).eq(7).all()
assert rea_stars.apply(lambda row: sum(row["days"]) == row["total"], axis=1).all(), (
    "Daily counts do not match weekly totals."
)

rea_stars["week_display_utc"] = pd.to_datetime(rea_stars["week"], unit="s", utc=True)
rea_stars = rea_stars.sort_values("week").reset_index(drop=True)

# Conservatively exclude the newest returned week.
# Its count may still be accumulating.
rea_stars["exclude_from_full_week_analysis"] = rea_stars["week"].eq(rea_stars["week"].max())

print("Pages collected:", len(rea_star_pages))
print("Weeks collected:", len(rea_stars))
display(rea_stars[["week_display_utc", "total", "exclude_from_full_week_analysis"]].tail(12))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

rea_complete = rea_stars.loc[~rea_stars["exclude_from_full_week_analysis"]].copy()

rea_anchor = rea_complete.loc[rea_complete["total"].idxmax()]
rea_anchor_week = rea_anchor["week_display_utc"]

# Compare the surge against the preceding four complete weeks.
rea_baseline = rea_complete.loc[
    rea_complete["week_display_utc"].ge(rea_anchor_week - pd.Timedelta(weeks=4))
    & rea_complete["week_display_utc"].lt(rea_anchor_week)
]
expected_baseline = pd.date_range(rea_anchor_week - pd.Timedelta(weeks=4), periods=4, freq="7D")
assert set(rea_baseline["week_display_utc"]) == set(expected_baseline), (
    "Baseline has missing weeks; inspect before comparing."
)

baseline_median = rea_baseline["total"].median()
surge_multiple = rea_anchor["total"] / baseline_median if baseline_median > 0 else float("nan")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

for ax, scale in zip(axes, ["linear", "symlog"], strict=False):
    colors = ["#f59e0b" if partial else "#2563eb" for partial in rea_stars["exclude_from_full_week_analysis"]]
    ax.bar(rea_stars["week_display_utc"], rea_stars["total"], width=5, color=colors)
    ax.set_yscale(scale)
    ax.set_title("Weekly stars: size of the surge" if scale == "linear" else "Weekly stars: earlier growth is visible")
    ax.set_xlabel("Week timestamp (displayed in UTC)")
    ax.set_ylabel("Stars gained")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.2)

axes[0].yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
fig.suptitle(
    "REA: sudden attention after months of modest star growth\nBlue = completed weeks | Orange = unfinished week"
)
fig.tight_layout()
plt.show()

print("Largest completed week:", rea_anchor_week)
print("Stars in that week:", f"{int(rea_anchor['total']):,}")
print("Preceding four-week median:", baseline_median)
print("Surge / baseline median:", f"{surge_multiple:,.1f}x")
print("Complete follow-up weeks:", int(rea_complete["week_display_utc"].gt(rea_anchor_week).sum()))

In [ ]:
# Retrieve REA's weekly commit activity.
rea_commit_response = rea_session.get(
    f"https://api.github.com/repos/{REA_REPO}/stats/commit_activity",
    headers={"Accept": "application/vnd.github+json"},
    timeout=30,
)

print("HTTP status:", rea_commit_response.status_code)

if rea_commit_response.status_code == 202:
    print("GitHub is preparing the statistics. Rerun this cell shortly.")
else:
    rea_commit_response.raise_for_status()
    rea_commit_payload = rea_commit_response.json()

    assert isinstance(rea_commit_payload, list), "Unexpected response."
    rea_commits = pd.DataFrame(rea_commit_payload)
    assert not rea_commits.empty, "No commit statistics returned."
    assert {"week", "total", "days"}.issubset(rea_commits.columns)
    assert not rea_commits["week"].duplicated().any()
    assert rea_commits["days"].apply(len).eq(7).all()
    assert rea_commits.apply(lambda row: sum(row["days"]) == row["total"], axis=1).all(), (
        "Daily counts do not match weekly totals."
    )

    rea_commits["week_display_utc"] = pd.to_datetime(rea_commits["week"], unit="s", utc=True)

    # Match exact source week timestamps. Missing data stays missing.
    rea_activity = rea_stars.rename(columns={"total": "weekly_stars"}).merge(
        rea_commits[["week", "total"]].rename(columns={"total": "weekly_commits"}),
        on="week",
        how="left",
        validate="one_to_one",
    )

    rea_activity["weeks_from_surge"] = (rea_activity["week_display_utc"] - rea_anchor_week).dt.days // 7

    display(
        rea_activity.loc[
            rea_activity["weeks_from_surge"].between(-4, 1),
            [
                "weeks_from_surge",
                "week_display_utc",
                "weekly_stars",
                "weekly_commits",
                "exclude_from_full_week_analysis",
            ],
        ]
    )

In [ ]:
rea_plot = rea_activity.loc[rea_activity["weeks_from_surge"].between(-4, 1)].copy()

assert rea_plot["weekly_commits"].notna().all(), "Missing commit coverage; inspect before plotting."

labels = rea_plot["week_display_utc"].dt.strftime("%b %d")
colors = ["#f59e0b" if partial else "#2563eb" for partial in rea_plot["exclude_from_full_week_analysis"]]

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

for ax, field, title in [
    (axes[0], "weekly_stars", "Attention: weekly stars gained"),
    (axes[1], "weekly_commits", "Development: weekly commits counted"),
]:
    bars = ax.bar(labels, rea_plot[field], color=colors)
    ax.bar_label(
        bars,
        labels=[f"{int(value):,}" for value in rea_plot[field]],
        padding=4,
        fontsize=10,
    )
    ax.set_title(title, loc="left")
    ax.set_ylim(0, rea_plot[field].max() * 1.25)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

axes[1].set_xlabel("Week timestamp displayed in UTC")
fig.suptitle(
    "REA: development ramped before the largest completed star surge\n"
    "Blue = completed weeks | Orange = unfinished week",
    fontsize=13,
)
fig.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path

assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive before saving."

rea_saved_at = pd.Timestamp.now(tz="UTC")
rea_save_dir = Path("/content/drive/MyDrive/Microsoft_3B/temporal/morluto_rea") / rea_saved_at.strftime(
    "%Y%m%dT%H%M%S%fZ"
)
rea_save_dir.mkdir(parents=True, exist_ok=False)

raw_evidence = {
    "repository": REA_REPO,
    "metadata_collected_at": rea_collected_at.isoformat(),
    "saved_at": rea_saved_at.isoformat(),
    "metadata": rea_metadata,
    "star_history_pages": rea_star_pages,
    "commit_statistics": rea_commit_payload,
    "commit_statistics_url": rea_commit_response.url,
    "notes": [
        "Newest returned star week excluded from complete-week analysis.",
        "No complete follow-up weeks after the October 4 surge.",
        "Weekly commit statistics are an activity proxy, not code quality.",
        "Star and commit responses were collected at different times.",
        "Commit collection methods across earlier pilots remain provisional.",
    ],
}

(rea_save_dir / "raw_evidence.json").write_text(json.dumps(raw_evidence, indent=2), encoding="utf-8")
rea_activity.to_csv(rea_save_dir / "weekly_activity.csv", index=False)

# 'fig' is the aligned stars-and-commits chart from the previous cell.
fig.savefig(rea_save_dir / "stars_and_commits.png", dpi=180, bbox_inches="tight")

print("Saved to:", rea_save_dir)
print("Files:", [p.name for p in rea_save_dir.iterdir()])

In [ ]:
# Fetch a small, recent sample from the completed surge week.
rea_sample_start = rea_anchor_week
rea_sample_end = rea_sample_start + pd.Timedelta(weeks=1)

rea_sample_response = rea_session.get(
    f"https://api.github.com/repos/{REA_REPO}/commits",
    headers={"Accept": "application/vnd.github+json"},
    params={
        "sha": rea_metadata["default_branch"],
        "since": rea_sample_start.isoformat(),
        "until": rea_sample_end.isoformat(),
        "per_page": 20,
    },
    timeout=30,
)
rea_sample_response.raise_for_status()
rea_sample_payload = rea_sample_response.json()

rea_commit_sample = pd.DataFrame(
    [
        {
            "sha": commit["sha"][:12],
            "committer_date": commit["commit"]["committer"]["date"],
            "message": commit["commit"]["message"].splitlines()[0],
            "is_merge": len(commit["parents"]) > 1,
            "url": commit["html_url"],
        }
        for commit in rea_sample_payload
    ]
)

assert not rea_commit_sample.empty, "No sample returned."
rea_commit_sample["committer_date"] = pd.to_datetime(rea_commit_sample["committer_date"], utc=True)

# Enforce our start-inclusive, end-exclusive window.
rea_commit_sample = rea_commit_sample.loc[
    rea_commit_sample["committer_date"].ge(rea_sample_start) & rea_commit_sample["committer_date"].lt(rea_sample_end)
].copy()

display(rea_commit_sample[["sha", "committer_date", "message", "is_merge"]])

# Preserve the response and the review table.
(rea_save_dir / "commit_sample_raw.json").write_text(
    json.dumps(
        {
            "request_url": rea_sample_response.url,
            "collected_at": pd.Timestamp.now(tz="UTC").isoformat(),
            "sampling_note": "First page, up to 20 commits; not random or exhaustive.",
            "response": rea_sample_payload,
        },
        indent=2,
    ),
    encoding="utf-8",
)
rea_commit_sample.to_csv(rea_save_dir / "commit_sample.csv", index=False)

In [ ]:
# Inspect the actual changes behind the first sampled commit.
commit_url = rea_sample_payload[0]["url"]

rea_detail_response = rea_session.get(
    commit_url,
    headers={"Accept": "application/vnd.github+json"},
    timeout=30,
)
rea_detail_response.raise_for_status()
rea_commit_detail = rea_detail_response.json()

print("Message:", rea_commit_detail["commit"]["message"])
print("Change counts:", rea_commit_detail["stats"])

display(
    pd.DataFrame(
        [
            {
                "file": item["filename"],
                "status": item["status"],
                "additions": item["additions"],
                "deletions": item["deletions"],
            }
            for item in rea_commit_detail.get("files", [])
        ]
    )
)

# Show bounded excerpts; patches can be omitted or truncated by GitHub.
for item in rea_commit_detail.get("files", [])[:3]:
    print("\nFILE:", item["filename"])
    print(item.get("patch", "Patch unavailable.")[:4000])

(rea_save_dir / "sample_commit_detail.json").write_text(
    json.dumps(rea_commit_detail, indent=2),
    encoding="utf-8",
)

## 12. What we can say—and what still needs work

**OpenClaw:** all eight follow-up weeks had commits. Median weekly commits rose from 1,570 before the peak to 2,110 afterward (about 1.34 times the baseline).

**DeepSeek-R1:** four of eight follow-up weeks had counted commits, with a median of 1.5. Its pre/post ratio is unavailable because the repository was too new for the baseline. A model-release repository may have different development needs from an application, so lower commit activity alone is not evidence of artificial virality.

**The methods are not interchangeable yet.** In one OpenClaw week (November 30, 2025), the weekly statistics reported 252 commits, while the individual-commit method counted 249 non-merge commits using either author or committer dates. That small difference is recorded, not explained. One checked week does not validate all weeks. Keep comparisons provisional until both pilots use one documented method.

The peak is the largest complete week in the collected history, not necessarily the start of virality. Collection coverage and repository age affect what is observable. These pilot results do not prove that attention was organic or manipulated.

### Progress and remaining work

- [x] Repair and freeze the 166-repository snapshot.
- [x] Collect and save the OpenClaw temporal pilot.
- [x] Collect DeepSeek-R1's historical event window.
- [x] Make the DeepSeek baseline creation-aware.
- [x] Record the commit-method discrepancy.
- [x] Consolidate the notebook into readable analysis sections.
- [ ] Run this cleaned notebook in Colab and confirm it loads the saved CSVs.
- [ ] Standardize commit collection across both pilots and verify coverage.
- [ ] Curate additional viral and comparison repositories with documented inclusion evidence, coordinating with Task #4.
- [ ] Add post-spike PR, issue, release, and fork activity; coordinate contributor measures with Task #2.
- [ ] Deliver documented temporal features and findings to Vidal's shared ML pipeline.

**Next step only:** open this notebook in Colab and run it from the top. Check the source messages in Section 9. No new collection is needed for this review.

### Saved collection records

The Drive folders shown in Section 9 contain the original CSVs and raw responses. OpenClaw's folder also contains `commit_method_check.json` and individual records. DeepSeek's folder contains `repository_metadata.json` and `pilot_features_creation_checked.csv`. The earlier uploaded notebook preserves the collection and debugging sequence in its version history. This cleaned version focuses on reviewing the saved evidence.
